# 🔍 Busca Heurística
## Melhor Custo, Melhor Estimativa e A*
### Inteligência Artificial — Ciência da Computação

---

**Conteúdo deste notebook:**
1. Introdução à Busca Heurística
2. Busca de Custo Uniforme (Uniform Cost Search)
3. Busca Gulosa (Greedy Best-First Search)
4. Algoritmo A* (A Estrela)
5. Comparação dos Algoritmos
6. Exemplo Prático: Navegação entre Cidades
7. Exercícios Propostos

## 1. Introdução à Busca Heurística

### O que é uma Heurística?

Uma **heurística** é uma "regra prática" ou uma estimativa que nos ajuda a tomar decisões mais inteligentes durante a busca por uma solução. A palavra vem do grego *heuriskein*, que significa "descobrir".

Na busca em grafos e árvores, uma **função heurística h(n)** fornece uma **estimativa do custo** para ir de um nó `n` até o objetivo. Por exemplo:
- Em um mapa de cidades, a **distância em linha reta** até o destino é uma boa heurística.
- Em um quebra-cabeça, o **número de peças fora do lugar** pode ser uma heurística.

### Por que precisamos de Busca Informada?

Na aula anterior, vimos as **buscas cegas** (busca em largura, busca em profundidade). Essas buscas exploram o espaço de estados **sem nenhuma informação** sobre quão perto estamos do objetivo. Isso funciona, mas pode ser muito **ineficiente** em problemas grandes.

A **busca informada** (ou busca heurística) utiliza **conhecimento adicional** sobre o problema para guiar a exploração de forma mais inteligente, expandindo primeiro os nós que parecem mais promissores.

### Diferença entre Busca Cega e Busca Heurística

| Característica | Busca Cega | Busca Heurística |
|---|---|---|
| Informação utilizada | Apenas a estrutura do grafo | Estrutura + estimativa de custo |
| Eficiência | Pode explorar muitos nós desnecessários | Tende a encontrar a solução mais rápido |
| Exemplos | Busca em Largura, Profundidade | Custo Uniforme, Gulosa, A* |

### Funções importantes

- **g(n)**: custo real do caminho desde o nó inicial até o nó `n`
- **h(n)**: estimativa (heurística) do custo de `n` até o objetivo
- **f(n)**: função de avaliação total. Em A*, temos `f(n) = g(n) + h(n)`

### Grafo de Exemplo

Ao longo deste notebook, usaremos o seguinte grafo ponderado para ilustrar os algoritmos:

```
        S ---1--- A ---12--- D
        |       / |          |
        4     2   5          3
        |   /     |          |
        B ---2--- C ---7--- G (objetivo)
```

**Custos das arestas:**
- S → A: 1,  S → B: 4
- A → B: 2,  A → C: 5,  A → D: 12
- B → C: 2
- C → D: 3,  C → G: 7
- D → G: 3

**Valores heurísticos h(n)** (estimativa da distância até G):

| Nó | h(n) |
|----|------|
| S  | 7    |
| A  | 6    |
| B  | 4    |
| C  | 2    |
| D  | 3    |
| G  | 0    |

Vamos usar este grafo para comparar o comportamento de cada algoritmo.

In [ ]:
# Definição do grafo de exemplo que será usado ao longo do notebook
# Cada nó mapeia para uma lista de tuplas (vizinho, custo)

grafo_exemplo = {
    'S': [('A', 1), ('B', 4)],
    'A': [('S', 1), ('B', 2), ('C', 5), ('D', 12)],
    'B': [('S', 4), ('A', 2), ('C', 2)],
    'C': [('A', 5), ('B', 2), ('D', 3), ('G', 7)],
    'D': [('A', 12), ('C', 3), ('G', 3)],
    'G': [('C', 7), ('D', 3)]
}

# Valores heurísticos: estimativa de distância até o objetivo G
heuristica_exemplo = {
    'S': 7,
    'A': 6,
    'B': 4,
    'C': 2,
    'D': 3,
    'G': 0
}

print("Grafo definido com sucesso!")
print("\nNós do grafo:", list(grafo_exemplo.keys()))
print("\nConexões de cada nó:")
for no, vizinhos in grafo_exemplo.items():
    conexoes = [f"{v}(custo={c})" for v, c in vizinhos]
    print(f"  {no} → {', '.join(conexoes)}")
print("\nHeurísticas:", heuristica_exemplo)

## 2. Busca de Custo Uniforme (Uniform Cost Search)

### Conceito

A **Busca de Custo Uniforme** (UCS, ou *Uniform Cost Search* - UCS) é um algoritmo que sempre expande o nó com o **menor custo de caminho g(n)** desde o nó inicial.

> **Ideia principal:** "Vamos sempre expandir o nó que tem o menor custo acumulado até agora."

### Como funciona?

1. Começamos com o nó inicial na **fila de prioridade** (ordenada pelo custo `g(n)`)
2. Removemos o nó com **menor custo** da fila
3. Se esse nó é o **objetivo**, retornamos o caminho
4. Caso contrário, **expandimos** o nó: adicionamos seus vizinhos à fila com o custo atualizado
5. Repetimos até encontrar o objetivo ou a fila ficar vazia

### Propriedades

| Propriedade | Valor |
|---|---|
| **Completo?** | ✅ Sim (se custos > 0) |
| **Ótimo?** | ✅ Sim (encontra o caminho de menor custo) |
| **Usa heurística?** | ❌ Não (usa apenas g(n)) |
| **Estrutura de dados** | Fila de prioridade (min-heap) |

### Passo a passo com nosso grafo

Objetivo: ir de **S** até **G**

| Passo | Nó expandido | g(n) | Fila de prioridade (nó, custo) |
|-------|-------------|------|-------------------------------|
| 1 | S | 0 | [(A,1), (B,4)] |
| 2 | A | 1 | [(B,3), (B,4), (C,6), (D,13)] |
| 3 | B (via A) | 3 | [(B,4), (C,5), (C,6), (D,13)] |
| 4 | B (via S)* | 4 | [(C,5), (C,6), (D,13)] |
| 5 | C (via B) | 5 | [(C,6), (D,8), (D,13), (G,12)] |
| 6 | C (via A)* | 6 | [(D,8), (D,13), (G,12)] |
| 7 | D (via C) | 8 | [(D,13), (G,11), (G,12)] |
| 8 | G (via D) | 11 | — |

*\* Nós já visitados são ignorados.*

**Resultado:** S → A → B → C → D → G, custo = **11**

Note que o UCS encontrou o caminho **ótimo** (menor custo), mas explorou muitos nós!

In [ ]:
import heapq  # Módulo para fila de prioridade (min-heap)

def busca_custo_uniforme(grafo, inicio, objetivo):
    """
    Busca de Custo Uniforme (Uniform Cost Search).
    
    Parâmetros:
        grafo: dicionário onde cada chave é um nó e o valor é uma lista
               de tuplas (vizinho, custo)
        inicio: nó inicial
        objetivo: nó que queremos alcançar
    
    Retorna:
        tupla (caminho, custo_total) ou (None, infinito) se não encontrar
    """
    # Fila de prioridade: (custo_acumulado, nó_atual, caminho_percorrido)
    fila = [(0, inicio, [inicio])]
    
    # Conjunto de nós já visitados (para não revisitar)
    visitados = set()
    
    # Contador de nós explorados (para análise)
    nos_explorados = 0
    
    while fila:
        # Remove o nó com menor custo da fila
        custo_atual, no_atual, caminho = heapq.heappop(fila)
        
        # Se já visitamos este nó, pulamos
        if no_atual in visitados:
            continue
        
        # Marca como visitado
        visitados.add(no_atual)
        nos_explorados += 1
        
        print(f"  Explorando: {no_atual} (custo acumulado: {custo_atual})")
        
        # Verificamos se chegamos ao objetivo
        if no_atual == objetivo:
            print(f"\n  ✅ Objetivo encontrado! Nós explorados: {nos_explorados}")
            return caminho, custo_atual
        
        # Expandimos os vizinhos do nó atual
        for vizinho, custo_aresta in grafo[no_atual]:
            if vizinho not in visitados:
                novo_custo = custo_atual + custo_aresta
                novo_caminho = caminho + [vizinho]
                heapq.heappush(fila, (novo_custo, vizinho, novo_caminho))
    
    # Se a fila esvaziou sem encontrar o objetivo
    print("  ❌ Objetivo não encontrado!")
    return None, float('inf')


# === Teste com nosso grafo de exemplo ===
print("=" * 50)
print("BUSCA DE CUSTO UNIFORME")
print("Objetivo: encontrar caminho de S até G")
print("=" * 50)

caminho, custo = busca_custo_uniforme(grafo_exemplo, 'S', 'G')

print(f"\nCaminho encontrado: {' → '.join(caminho)}")
print(f"Custo total: {custo}")

## 3. Busca Gulosa (Greedy Best-First Search)

### Conceito

A **Busca Gulosa** (ou *Greedy Best-First Search*) é um algoritmo que sempre expande o nó que parece estar **mais perto do objetivo**, de acordo com a função heurística **h(n)**.

> **Ideia principal:** "Vamos sempre expandir o nó que *parece* mais promissor segundo nossa estimativa."

### Como funciona?

1. Começamos com o nó inicial na fila de prioridade (ordenada por **h(n)**)
2. Removemos o nó com **menor h(n)** da fila
3. Se esse nó é o objetivo, retornamos o caminho
4. Caso contrário, expandimos o nó: adicionamos seus vizinhos à fila
5. Repetimos até encontrar o objetivo ou a fila ficar vazia

### Diferença para o Custo Uniforme

- **Custo Uniforme** ordena pela função **g(n)** → custo real acumulado
- **Busca Gulosa** ordena pela função **h(n)** → estimativa até o objetivo

### Propriedades

| Propriedade | Valor |
|---|---|
| **Completo?** | ❌ Não necessariamente (pode entrar em loops em grafos sem controle) |
| **Ótimo?** | ❌ Não (a heurística pode enganar) |
| **Usa heurística?** | ✅ Sim (usa apenas h(n)) |
| **Vantagem** | Geralmente é muito rápida |
| **Estrutura de dados** | Fila de prioridade (min-heap) |

### ⚠️ Por que a Busca Gulosa não é ótima?

A busca gulosa pode ser "enganada" pela heurística. Ela escolhe o nó que **parece** mais perto do objetivo, mas o caminho real pode ser mais longo. É como escolher sempre a estrada que aponta na direção do destino, mesmo que haja um atalho por outro caminho.

### Passo a passo com nosso grafo

Objetivo: ir de **S** até **G**, usando h(n)

| Passo | Nó expandido | h(n) | Fila de prioridade (nó, h(n)) |
|-------|-------------|------|-------------------------------|
| 1 | S | 7 | [(A,6), (B,4)] |
| 2 | B | 4 | [(A,6), (C,2)] |
| 3 | C | 2 | [(A,6), (D,3), (G,0)] |
| 4 | G | 0 | — |

**Resultado:** S → B → C → G, custo real = 4 + 2 + 7 = **13**

Note que a busca gulosa encontrou o caminho rápido (poucos nós explorados), mas **não encontrou o caminho ótimo** (que custaria 11)!

In [ ]:
import heapq

def busca_gulosa(grafo, inicio, objetivo, heuristica):
    """
    Busca Gulosa (Greedy Best-First Search).
    
    Parâmetros:
        grafo: dicionário representando o grafo ponderado
        inicio: nó inicial
        objetivo: nó que queremos alcançar
        heuristica: dicionário com valores h(n) para cada nó
    
    Retorna:
        tupla (caminho, custo_total) ou (None, infinito) se não encontrar
    """
    # Fila de prioridade: (heurística_h(n), nó_atual, caminho, custo_real)
    fila = [(heuristica[inicio], inicio, [inicio], 0)]
    
    # Conjunto de nós já visitados
    visitados = set()
    
    # Contador de nós explorados
    nos_explorados = 0
    
    while fila:
        # Remove o nó com menor h(n) da fila
        h_atual, no_atual, caminho, custo_real = heapq.heappop(fila)
        
        # Se já visitamos este nó, pulamos
        if no_atual in visitados:
            continue
        
        # Marca como visitado
        visitados.add(no_atual)
        nos_explorados += 1
        
        print(f"  Explorando: {no_atual} (h={h_atual}, custo real acumulado={custo_real})")
        
        # Verificamos se chegamos ao objetivo
        if no_atual == objetivo:
            print(f"\n  ✅ Objetivo encontrado! Nós explorados: {nos_explorados}")
            return caminho, custo_real
        
        # Expandimos os vizinhos
        for vizinho, custo_aresta in grafo[no_atual]:
            if vizinho not in visitados:
                novo_custo = custo_real + custo_aresta
                novo_caminho = caminho + [vizinho]
                heapq.heappush(fila, (heuristica[vizinho], vizinho, 
                                       novo_caminho, novo_custo))
    
    print("  ❌ Objetivo não encontrado!")
    return None, float('inf')


# === Teste com nosso grafo de exemplo ===
print("=" * 50)
print("BUSCA GULOSA (GREEDY BEST-FIRST)")
print("Objetivo: encontrar caminho de S até G")
print("=" * 50)

caminho, custo = busca_gulosa(grafo_exemplo, 'S', 'G', heuristica_exemplo)

print(f"\nCaminho encontrado: {' → '.join(caminho)}")
print(f"Custo total do caminho: {custo}")
print(f"\n⚠️  Note que o custo ({custo}) é MAIOR que o ótimo (11)!")
print("A busca gulosa é rápida, mas não garante o melhor caminho.")

## 4. Algoritmo A* (A Estrela)

### Conceito

O **Algoritmo A*** é considerado um dos algoritmos de busca mais importantes da Inteligência Artificial. Ele combina o melhor dos dois mundos:
- O **custo real** g(n) da Busca de Custo Uniforme
- A **estimativa heurística** h(n) da Busca Gulosa

> **Ideia principal:** "Vamos expandir o nó que minimiza o custo total estimado f(n) = g(n) + h(n)"

Onde:
- **g(n)** = custo real do caminho desde o início até o nó n
- **h(n)** = estimativa do custo de n até o objetivo
- **f(n) = g(n) + h(n)** = estimativa do custo total do melhor caminho passando por n

### Como funciona?

1. Começamos com o nó inicial na fila de prioridade (ordenada por **f(n) = g(n) + h(n)**)
2. Removemos o nó com **menor f(n)** da fila
3. Se esse nó é o objetivo, retornamos o caminho
4. Caso contrário, expandimos o nó: para cada vizinho, calculamos o novo f(n)
5. Repetimos até encontrar o objetivo ou a fila ficar vazia

### Propriedades

| Propriedade | Valor |
|---|---|
| **Completo?** | ✅ Sim |
| **Ótimo?** | ✅ Sim (com heurística admissível) |
| **Usa heurística?** | ✅ Sim (usa g(n) + h(n)) |
| **Estrutura de dados** | Fila de prioridade (min-heap) |

### 🔑 Heurística Admissível

Para que o A* encontre o caminho ótimo, a heurística deve ser **admissível**:

> Uma heurística h(n) é **admissível** se ela **nunca superestima** o custo real para alcançar o objetivo. Ou seja, h(n) ≤ custo_real(n, objetivo) para todo nó n.

**Exemplo:** A distância em linha reta entre duas cidades é uma heurística admissível, porque a distância real pela estrada é sempre ≥ à distância em linha reta.

### 🔑 Heurística Consistente (Monotônica)

Uma heurística é **consistente** se, para todo nó n e todo sucessor n' de n:

> h(n) ≤ custo(n, n') + h(n')

Isso é como a **desigualdade triangular**. Toda heurística consistente é admissível, mas nem toda admissível é consistente.

### Passo a passo com nosso grafo

Objetivo: ir de **S** até **G**, usando f(n) = g(n) + h(n)

| Passo | Nó | g(n) | h(n) | f(n)=g+h | Ação |
|-------|-----|------|------|----------|------|
| 1 | S | 0 | 7 | 7 | Expande S. Adiciona A(f=1+6=7), B(f=4+4=8) |
| 2 | A | 1 | 6 | 7 | Expande A. Adiciona B(f=3+4=7), C(f=6+2=8), D(f=13+3=16) |
| 3 | B (via A) | 3 | 4 | 7 | Expande B. Adiciona C(f=5+2=7) |
| 4 | C (via B) | 5 | 2 | 7 | Expande C. Adiciona D(f=8+3=11), G(f=12+0=12) |
| 5 | B (via S)* | 4 | 4 | 8 | Já visitado, ignora |
| 6 | C (via A)* | 6 | 2 | 8 | Já visitado, ignora |
| 7 | D | 8 | 3 | 11 | Expande D. Adiciona G(f=11+0=11) |
| 8 | G | 11 | 0 | 11 | ✅ Objetivo encontrado! |

*\* Nós já visitados são ignorados.*

**Resultado:** S → A → B → C → D → G, custo = **11** (ótimo!)

O A* encontrou o caminho ótimo, assim como a Busca de Custo Uniforme, mas de forma potencialmente mais eficiente graças à heurística!

In [ ]:
import heapq

def busca_a_estrela(grafo, inicio, objetivo, heuristica):
    """
    Algoritmo A* (A Estrela).
    
    Parâmetros:
        grafo: dicionário representando o grafo ponderado
        inicio: nó inicial
        objetivo: nó que queremos alcançar
        heuristica: dicionário com valores h(n) para cada nó
    
    Retorna:
        tupla (caminho, custo_total) ou (None, infinito) se não encontrar
    """
    # Fila de prioridade: (f(n), g(n), nó_atual, caminho)
    # f(n) = g(n) + h(n) é usado para ordenar a fila
    g_inicio = 0
    f_inicio = g_inicio + heuristica[inicio]
    fila = [(f_inicio, g_inicio, inicio, [inicio])]
    
    # Conjunto de nós já visitados
    visitados = set()
    
    # Contador de nós explorados
    nos_explorados = 0
    
    while fila:
        # Remove o nó com menor f(n) da fila
        f_atual, g_atual, no_atual, caminho = heapq.heappop(fila)
        
        # Se já visitamos este nó, pulamos
        if no_atual in visitados:
            continue
        
        # Marca como visitado
        visitados.add(no_atual)
        nos_explorados += 1
        
        print(f"  Explorando: {no_atual} (g={g_atual}, h={heuristica[no_atual]}, f={f_atual})")
        
        # Verificamos se chegamos ao objetivo
        if no_atual == objetivo:
            print(f"\n  ✅ Objetivo encontrado! Nós explorados: {nos_explorados}")
            return caminho, g_atual
        
        # Expandimos os vizinhos
        for vizinho, custo_aresta in grafo[no_atual]:
            if vizinho not in visitados:
                g_novo = g_atual + custo_aresta
                f_novo = g_novo + heuristica[vizinho]
                novo_caminho = caminho + [vizinho]
                heapq.heappush(fila, (f_novo, g_novo, vizinho, novo_caminho))
    
    print("  ❌ Objetivo não encontrado!")
    return None, float('inf')


# === Teste com nosso grafo de exemplo ===
print("=" * 50)
print("ALGORITMO A* (A ESTRELA)")
print("Objetivo: encontrar caminho de S até G")
print("=" * 50)

caminho, custo = busca_a_estrela(grafo_exemplo, 'S', 'G', heuristica_exemplo)

print(f"\nCaminho encontrado: {' → '.join(caminho)}")
print(f"Custo total do caminho: {custo}")
print(f"\n✅ A* encontrou o caminho ótimo!")

## 5. Comparação dos Algoritmos de Busca

### Tabela Comparativa

| Característica | Custo Uniforme | Busca Gulosa | A* |
|---|---|---|---|
| **Função de ordenação** | g(n) | h(n) | f(n) = g(n) + h(n) |
| **Completo?** | ✅ Sim | ❌ Não sempre | ✅ Sim |
| **Ótimo?** | ✅ Sim | ❌ Não | ✅ Sim (h admissível) |
| **Usa heurística?** | ❌ Não | ✅ Sim | ✅ Sim |
| **Complexidade de tempo** | O(b^(1+⌊C*/ε⌋)) | O(b^m) | O(b^d) |
| **Complexidade de espaço** | O(b^(1+⌊C*/ε⌋)) | O(b^m) | O(b^d) |
| **Velocidade típica** | Lento | Rápido | Moderado |
| **Qualidade da solução** | Ótima | Pode ser ruim | Ótima |

Onde:
- **b** = fator de ramificação (número médio de filhos por nó)
- **d** = profundidade da solução ótima
- **m** = profundidade máxima da árvore
- **C*** = custo da solução ótima
- **ε** = menor custo de aresta

### Resumo Visual

```
Custo Uniforme:  Seguro, mas lento    → "Explora tudo com cuidado"
Busca Gulosa:    Rápido, mas arriscado → "Corre na direção do objetivo"  
A*:              Inteligente e seguro  → "Combina cautela com intuição"
```

### Quando usar cada um?

- **Custo Uniforme**: Quando não temos nenhuma heurística disponível, mas precisamos do caminho ótimo.
- **Busca Gulosa**: Quando precisamos de uma resposta rápida e a qualidade não precisa ser perfeita.
- **A***: Quando precisamos do caminho ótimo e temos uma boa heurística. É a escolha padrão na maioria dos problemas práticos.

In [ ]:
# Vamos comparar os três algoritmos no mesmo grafo!

print("=" * 60)
print("COMPARAÇÃO DOS TRÊS ALGORITMOS")
print("Grafo de exemplo: S até G")
print("=" * 60)

print("\n--- Busca de Custo Uniforme ---")
caminho_ucs, custo_ucs = busca_custo_uniforme(grafo_exemplo, 'S', 'G')

print("\n--- Busca Gulosa ---")
caminho_gulosa, custo_gulosa = busca_gulosa(grafo_exemplo, 'S', 'G', heuristica_exemplo)

print("\n--- Algoritmo A* ---")
caminho_astar, custo_astar = busca_a_estrela(grafo_exemplo, 'S', 'G', heuristica_exemplo)

# Resumo
print("\n" + "=" * 60)
print("RESUMO DA COMPARAÇÃO")
print("=" * 60)
print(f"{'Algoritmo':<25} {'Caminho':<25} {'Custo':<10}")
print("-" * 60)
print(f"{'Custo Uniforme':<25} {' → '.join(caminho_ucs):<25} {custo_ucs:<10}")
print(f"{'Busca Gulosa':<25} {' → '.join(caminho_gulosa):<25} {custo_gulosa:<10}")
print(f"{'A*':<25} {' → '.join(caminho_astar):<25} {custo_astar:<10}")
print("-" * 60)
print(f"\n🏆 Custo Uniforme e A* encontraram o caminho ótimo (custo {custo_ucs})!")
print(f"⚠️  A Busca Gulosa encontrou um caminho subótimo (custo {custo_gulosa}).")

## 6. Exemplo Prático: Navegação entre Cidades Brasileiras

Agora vamos aplicar os três algoritmos em um problema mais realista: encontrar o melhor caminho entre cidades brasileiras.

### Mapa de Cidades

Vamos usar um mapa simplificado com as seguintes cidades e distâncias rodoviárias (em km):

```
                    Brasília
                   /        \
                 740         210
                /              \
    Belo Horizonte --- 480 --- Goiânia
        |       \
       440       590
        |         \
  Rio de Janeiro   São Paulo
                   /       \
                 400        1000
                /             \
           Curitiba       Campo Grande
              |
             300
              |
         Florianópolis
              |
             460
              |
        Porto Alegre
```

**Objetivo:** Encontrar o melhor caminho de **Porto Alegre** até **Brasília**

### Heurísticas (distância em linha reta até Brasília, em km)

| Cidade | h(n) - Distância em linha reta |
|--------|------|
| Porto Alegre | 1620 |
| Florianópolis | 1400 |
| Curitiba | 1090 |
| São Paulo | 870 |
| Rio de Janeiro | 930 |
| Belo Horizonte | 620 |
| Goiânia | 175 |
| Campo Grande | 870 |
| Brasília | 0 |

> **Nota:** As distâncias em linha reta são **sempre menores ou iguais** às distâncias reais pela estrada, o que garante que nossa heurística é **admissível**!

In [ ]:
# === Definição do mapa de cidades brasileiras ===

mapa_brasil = {
    'Porto Alegre':    [('Florianópolis', 460), ('Curitiba', 710)],
    'Florianópolis':   [('Porto Alegre', 460), ('Curitiba', 300)],
    'Curitiba':        [('Porto Alegre', 710), ('Florianópolis', 300), ('São Paulo', 400)],
    'São Paulo':       [('Curitiba', 400), ('Rio de Janeiro', 430), 
                        ('Belo Horizonte', 590), ('Campo Grande', 1000)],
    'Rio de Janeiro':  [('São Paulo', 430), ('Belo Horizonte', 440)],
    'Belo Horizonte':  [('São Paulo', 590), ('Rio de Janeiro', 440), 
                        ('Brasília', 740), ('Goiânia', 480)],
    'Goiânia':         [('Belo Horizonte', 480), ('Brasília', 210), ('Campo Grande', 840)],
    'Campo Grande':    [('São Paulo', 1000), ('Goiânia', 840)],
    'Brasília':        [('Belo Horizonte', 740), ('Goiânia', 210)]
}

# Heurística: distância em linha reta até Brasília (em km)
h_brasilia = {
    'Porto Alegre':   1620,
    'Florianópolis':  1400,
    'Curitiba':       1090,
    'São Paulo':       870,
    'Rio de Janeiro':  930,
    'Belo Horizonte':  620,
    'Goiânia':         175,
    'Campo Grande':    870,
    'Brasília':          0
}

print("Mapa de cidades brasileiras carregado!")
print(f"\nCidades no mapa: {len(mapa_brasil)}")
for cidade, conexoes in mapa_brasil.items():
    vizinhos = [f"{v} ({d}km)" for v, d in conexoes]
    print(f"  📍 {cidade}: {', '.join(vizinhos)}")

In [ ]:
# === Comparação dos três algoritmos no mapa do Brasil ===

origem = 'Porto Alegre'
destino = 'Brasília'

print("=" * 65)
print(f"NAVEGAÇÃO: {origem} → {destino}")
print("=" * 65)

# 1. Busca de Custo Uniforme
print("\n" + "─" * 65)
print("📊 BUSCA DE CUSTO UNIFORME")
print("─" * 65)
caminho1, custo1 = busca_custo_uniforme(mapa_brasil, origem, destino)
print(f"Caminho: {' → '.join(caminho1)}")
print(f"Distância total: {custo1} km")

# 2. Busca Gulosa
print("\n" + "─" * 65)
print("📊 BUSCA GULOSA")
print("─" * 65)
caminho2, custo2 = busca_gulosa(mapa_brasil, origem, destino, h_brasilia)
print(f"Caminho: {' → '.join(caminho2)}")
print(f"Distância total: {custo2} km")

# 3. A*
print("\n" + "─" * 65)
print("📊 ALGORITMO A*")
print("─" * 65)
caminho3, custo3 = busca_a_estrela(mapa_brasil, origem, destino, h_brasilia)
print(f"Caminho: {' → '.join(caminho3)}")
print(f"Distância total: {custo3} km")

# Resumo final
print("\n" + "=" * 65)
print("📋 RESUMO FINAL")
print("=" * 65)
print(f"{'Algoritmo':<25} {'Distância (km)':<18} {'Cidades no caminho'}")
print("─" * 65)
print(f"{'Custo Uniforme':<25} {custo1:<18} {len(caminho1)}")
print(f"{'Busca Gulosa':<25} {custo2:<18} {len(caminho2)}")
print(f"{'A*':<25} {custo3:<18} {len(caminho3)}")
print("─" * 65)

# Identificar o melhor
melhor_custo = min(custo1, custo2, custo3)
print(f"\n🏆 Menor distância encontrada: {melhor_custo} km")
if custo1 == melhor_custo:
    print("   → Custo Uniforme encontrou o caminho ótimo")
if custo3 == melhor_custo:
    print("   → A* encontrou o caminho ótimo")
if custo2 > melhor_custo:
    print(f"   → Busca Gulosa encontrou caminho {custo2 - melhor_custo} km mais longo")

In [ ]:
# === Testando com outra origem ===

print("=" * 65)
print("TESTE ADICIONAL: Campo Grande → Brasília")
print("=" * 65)

origem2 = 'Campo Grande'
destino2 = 'Brasília'

print("\n--- Custo Uniforme ---")
c1, k1 = busca_custo_uniforme(mapa_brasil, origem2, destino2)
print(f"Resultado: {' → '.join(c1)}, custo = {k1} km\n")

print("--- Busca Gulosa ---")
c2, k2 = busca_gulosa(mapa_brasil, origem2, destino2, h_brasilia)
print(f"Resultado: {' → '.join(c2)}, custo = {k2} km\n")

print("--- A* ---")
c3, k3 = busca_a_estrela(mapa_brasil, origem2, destino2, h_brasilia)
print(f"Resultado: {' → '.join(c3)}, custo = {k3} km")

print("\n📌 Observe como os diferentes algoritmos podem encontrar")
print("   caminhos diferentes dependendo do grafo e da heurística!")

## 7. A Importância da Qualidade da Heurística

### Heurística perfeita vs. heurística ruim

A qualidade da heurística afeta diretamente o desempenho do A*:

- **h(n) = 0 para todo n**: O A* se comporta exatamente como a Busca de Custo Uniforme (pois f(n) = g(n) + 0 = g(n)). É admissível, mas não ajuda a guiar a busca.

- **h(n) = h*(n)** (heurística perfeita): O A* vai direto para a solução sem explorar nenhum nó desnecessário. Mas geralmente não conhecemos h*(n), senão já teríamos a solução!

- **h(n) > h*(n)** (heurística superestimadora): O A* pode **não** encontrar a solução ótima, pois a heurística não é admissível.

### Dominância entre heurísticas

Se temos duas heurísticas admissíveis h₁ e h₂, e para todo nó n:

> h₂(n) ≥ h₁(n)

Dizemos que **h₂ domina h₁**. Uma heurística dominante é **sempre melhor** (ou pelo menos igual) em termos de nós explorados.

### Dica prática

Na hora de criar uma heurística:
1. ✅ Garanta que ela seja **admissível** (nunca superestime)
2. ✅ Faça ela ser o mais **informativa possível** (maior valor, sem superestimar)
3. ✅ Garanta que seja **fácil de calcular** (não pode ser mais custosa que resolver o problema!)

In [ ]:
# === Demonstrando o efeito da qualidade da heurística ===

print("=" * 65)
print("EFEITO DA QUALIDADE DA HEURÍSTICA NO A*")
print("Grafo: S até G")
print("=" * 65)

# Heurística 1: h(n) = 0 (equivalente a Custo Uniforme)
h_zero = {no: 0 for no in grafo_exemplo}
print("\n🔹 Heurística h(n) = 0 (sem informação):")
c1, k1 = busca_a_estrela(grafo_exemplo, 'S', 'G', h_zero)
print(f"   Caminho: {' → '.join(c1)}, custo: {k1}")

# Heurística 2: nossa heurística original (admissível)
print("\n🔹 Heurística admissível (nossa original):")
c2, k2 = busca_a_estrela(grafo_exemplo, 'S', 'G', heuristica_exemplo)
print(f"   Caminho: {' → '.join(c2)}, custo: {k2}")

# Heurística 3: superestimadora (NÃO admissível!)
h_super = {'S': 20, 'A': 15, 'B': 12, 'C': 10, 'D': 8, 'G': 0}
print("\n🔹 Heurística SUPERESTIMADORA (não admissível):")
c3, k3 = busca_a_estrela(grafo_exemplo, 'S', 'G', h_super)
print(f"   Caminho: {' → '.join(c3)}, custo: {k3}")

print("\n" + "─" * 65)
print("📌 Observações:")
print("• h=0 encontra o ótimo, mas explora mais nós")
print("• Heurística admissível encontra o ótimo com menos exploração")
print(f"• Heurística superestimadora pode dar resultado subótimo (custo={k3})")

## 8. Exercícios Propostos

### Exercício 1: Rastreando algoritmos manualmente

Considere o seguinte grafo:

```
    A ---3--- B ---4--- E
    |         |         |
    2         1         2
    |         |         |
    C ---5--- D ---6--- F (objetivo)
```

Heurísticas para F:
| Nó | h(n) |
|----|------|
| A  | 8    |
| B  | 5    |
| C  | 7    |
| D  | 6    |
| E  | 2    |
| F  | 0    |

**a)** Execute manualmente a **Busca de Custo Uniforme** de A até F. Mostre a fila de prioridade em cada passo.

**b)** Execute manualmente a **Busca Gulosa** de A até F. Qual caminho foi encontrado? É ótimo?

**c)** Execute manualmente o **A*** de A até F. Compare o resultado com os algoritmos anteriores.

---

### Exercício 2: Implementação — Outro mapa

Crie um grafo representando pelo menos **6 cidades** de um estado brasileiro que você conhece. Defina:
1. As distâncias rodoviárias entre as cidades (arestas do grafo)
2. As distâncias em linha reta até uma cidade-destino (heurística)

Aplique os três algoritmos e compare os resultados. A heurística que você usou é admissível? Justifique.

---

### Exercício 3: Heurística admissível?

Para cada função heurística abaixo, diga se é **admissível** para o problema de encontrar o caminho mais curto em um mapa de estradas, e justifique:

**a)** h(n) = distância em linha reta de n até o objetivo

**b)** h(n) = distância rodoviária real de n até o objetivo

**c)** h(n) = 2 × distância em linha reta de n até o objetivo

**d)** h(n) = 0 para todo nó n

---

### Exercício 4: Desafio — Labirinto

Implemente o A* para resolver um labirinto representado como uma grade (matriz). Use como heurística a **distância de Manhattan** (soma das diferenças absolutas das coordenadas):

> h(n) = |x_n - x_objetivo| + |y_n - y_objetivo|

Dica: represente o labirinto como uma lista de listas, onde 0 = caminho livre e 1 = parede. Cada posição (linha, coluna) é um nó do grafo.

```python
labirinto = [
    [0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
]
# Início: (0, 0), Objetivo: (4, 4)
```

In [ ]:
# === Grafo do Exercício 1 (para você testar suas respostas!) ===

grafo_ex1 = {
    'A': [('B', 3), ('C', 2)],
    'B': [('A', 3), ('D', 1), ('E', 4)],
    'C': [('A', 2), ('D', 5)],
    'D': [('B', 1), ('C', 5), ('F', 6)],
    'E': [('B', 4), ('F', 2)],
    'F': [('D', 6), ('E', 2)]
}

heuristica_ex1 = {
    'A': 8, 'B': 5, 'C': 7, 'D': 6, 'E': 2, 'F': 0
}

print("Exercício 1 — Confira suas respostas manuais:")
print("=" * 50)

print("\n--- a) Custo Uniforme: A → F ---")
cam1, cst1 = busca_custo_uniforme(grafo_ex1, 'A', 'F')
print(f"Caminho: {' → '.join(cam1)}, Custo: {cst1}")

print("\n--- b) Busca Gulosa: A → F ---")
cam2, cst2 = busca_gulosa(grafo_ex1, 'A', 'F', heuristica_ex1)
print(f"Caminho: {' → '.join(cam2)}, Custo: {cst2}")

print("\n--- c) A*: A → F ---")
cam3, cst3 = busca_a_estrela(grafo_ex1, 'A', 'F', heuristica_ex1)
print(f"Caminho: {' → '.join(cam3)}, Custo: {cst3}")

print("\n📌 Compare com suas respostas feitas à mão!")

## 📚 Referências

1. **Russell, S. & Norvig, P.** *Inteligência Artificial: Uma Abordagem Moderna*. 3ª ed. Editora Campus/Elsevier, 2013. — Capítulos 3 e 4.

2. **Luger, G. F.** *Inteligência Artificial*. 6ª ed. Editora Pearson, 2013. — Capítulo 3.

3. **Coppin, B.** *Inteligência Artificial*. Editora LTC, 2010.

---

**Notebook preparado para a disciplina de Inteligência Artificial.**
**Curso de Ciência da Computação.**